In [20]:
!pip install datasets

In [2]:
from huggingface_hub import login
login("Your HF Token")

In [36]:
import torch
from datasets import load_dataset
from torchvision import transforms
from torch.utils.data import DataLoader, TensorDataset

print("Loading ImageNet Validation Dataset (50,000 images)...")

# --------------------------------------------------
# Image Transformations
# --------------------------------------------------
transform = transforms.Compose([
    transforms.Resize(36),
    transforms.CenterCrop(32),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# --------------------------------------------------
# Load the dataset
# --------------------------------------------------
dataset = load_dataset(
    "parquet",
    data_files={
        "train": "hf://datasets/ILSVRC/imagenet-1k/data/validation-*.parquet"
    },
    split="train"
)

# --------------------------------------------------
# PRE-TRANSFORM EVERYTHING ONCE INTO A TENSOR
# --------------------------------------------------
print("Pre-processing all 50,000 images into RAM (one-time cost)...")

all_images = []
for i in range(len(dataset)):
    img = dataset[i]["image"].convert("RGB")
    all_images.append(transform(img))
    if (i + 1) % 10000 == 0:
        print(f"  Processed {i+1}/50000")

all_images_tensor = torch.stack(all_images)  # Shape: [50000, 3, 32, 32]
print(f"Done! Tensor shape: {all_images_tensor.shape}")
print(f"Memory: {all_images_tensor.nbytes / 1e9:.2f} GB")  # ~0.6 GB for 32x32

del dataset  # free the raw dataset

# --------------------------------------------------
# RandomHorizontalFlip during training (GPU-side)
# --------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
# Move entire dataset to GPU — it's only ~0.6 GB
all_images_gpu = all_images_tensor.to(device)
del all_images_tensor  # free CPU copy
num_batches = len(all_images_gpu) // batch_size  # 48
print(f"Success! {len(all_images_gpu)} images, {num_batches} batches. Data lives on GPU.")

Loading ImageNet Validation Dataset (50,000 images)...
Pre-processing all 50,000 images into RAM (one-time cost)...
  Processed 10000/50000
  Processed 20000/50000
  Processed 30000/50000
  Processed 40000/50000
  Processed 50000/50000
Done! Tensor shape: torch.Size([50000, 3, 32, 32])
Memory: 0.61 GB
Success! 50000 images, 48 batches. Data lives on GPU.


In [37]:
print(all_images_tensor.shape)
print(all_images_tensor.dtype)
print(len(train_loader), "batches")

NameError: name 'all_images_tensor' is not defined

In [17]:
import random

In [40]:
import math
import torch.nn as nn

# ==========================================
# 1. THE COSINE NOISE SCHEDULE
# (Replaces the linear schedule for vastly better middle-step learning)
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
timesteps = 1000
torch.backends.cudnn.benchmark = True

def get_cosine_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.02).to(device)

betas = get_cosine_schedule(timesteps)
alphas = 1.0 - betas
alpha_cumprod = torch.cumprod(alphas, dim=0).to(device)

def add_noise(images, noise, t, alpha_cumprod):
    sqrt_alpha_cumprod = torch.sqrt(alpha_cumprod[t])[:, None, None, None]
    sqrt_one_minus_alpha_cumprod = torch.sqrt(1 - alpha_cumprod[t])[:, None, None, None]
    return sqrt_alpha_cumprod * images + sqrt_one_minus_alpha_cumprod * noise

# ==========================================
# 2. UPGRADED UNET COMPONENTS
# ==========================================
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=time.device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

class ResNetBlock(nn.Module):
    # UPGRADE: Injected Dropout (0.1)
    def __init__(self, in_channels, out_channels, time_emb_dim, groups=8, dropout=0.1):
        super().__init__()
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_channels))
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.gn1 = nn.GroupNorm(groups, out_channels)
        self.act1 = nn.SiLU()
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.gn2 = nn.GroupNorm(groups, out_channels)
        self.act2 = nn.SiLU()
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t_emb):
        h = self.act1(self.gn1(self.conv1(x)))
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.dropout(h) # Dropout applied directly to the math stream
        h = self.act2(self.gn2(self.conv2(h)))
        return h + self.shortcut(x)

class SelfAttentionBlock(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=channels, num_heads=num_heads, batch_first=True)
        self.ln = nn.LayerNorm([channels])

    def forward(self, x):
        b, c, h, w = x.shape
        x_flat = x.view(b, c, h * w).transpose(1, 2)
        x_norm = self.ln(x_flat)
        attn_out, _ = self.mha(x_norm, x_norm, x_norm)
        return (attn_out + x_flat).transpose(1, 2).view(b, c, h, w)

# ==========================================
# 3. THE UPGRADED ARCHITECTURE
# ==========================================
class DDPMUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, time_emb_dim=256):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim), 
            nn.Linear(time_emb_dim, time_emb_dim), 
            nn.SiLU()
        )
        self.init_conv = nn.Conv2d(in_channels, 64, 3, padding=1)
        
        # DOWN PATH
        self.down1 = ResNetBlock(64, 64, time_emb_dim) 
        self.pool1 = nn.MaxPool2d(2) 
        
        self.down2 = ResNetBlock(64, 128, time_emb_dim) 
        self.attn_down = SelfAttentionBlock(128) # UPGRADE: Attention at 16x16
        self.pool2 = nn.MaxPool2d(2) 
        
        # BOTTLENECK (8x8)
        self.mid1 = ResNetBlock(128, 128, time_emb_dim)
        self.mid_attn = SelfAttentionBlock(128)
        self.mid2 = ResNetBlock(128, 128, time_emb_dim)
        
        # UP PATH
        self.upsample1 = nn.Upsample(scale_factor=2, mode='nearest') 
        self.up1 = ResNetBlock(128 + 128, 128, time_emb_dim) 
        self.attn_up = SelfAttentionBlock(128) # UPGRADE: Attention at 16x16
        
        self.upsample2 = nn.Upsample(scale_factor=2, mode='nearest') 
        self.up2 = ResNetBlock(128 + 64, 64, time_emb_dim) 
        
        self.final_conv = nn.Conv2d(64, out_channels, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        
        # Down
        x1 = self.init_conv(x)
        d1 = self.down1(x1, t_emb)      # 32x32, 64ch
        
        d2 = self.down2(self.pool1(d1), t_emb) # 16x16, 128ch
        d2 = self.attn_down(d2)
        
        # Bottleneck
        m = self.mid1(self.pool2(d2), t_emb)   # 8x8, 128ch
        m = self.mid_attn(m)
        m = self.mid2(m, t_emb)
        
        # Up
        u1 = self.upsample1(m)
        u1 = torch.cat([u1, d2], dim=1) # Concat mid (128) and d2 (128)
        u1 = self.up1(u1, t_emb)        # 16x16, 128ch
        u1 = self.attn_up(u1)
        
        u2 = self.upsample2(u1)
        u2 = torch.cat([u2, d1], dim=1) # Concat u1 (128) and d1 (64)
        u2 = self.up2(u2, t_emb)        # 32x32, 64ch
        
        return self.final_conv(u2)

model = DDPMUNet().to(device)
print(f"Model on: {next(model.parameters()).device}")
print(f"Upgraded Model Built! Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

Model on: cuda:0
Upgraded Model Built! Total Parameters: 1,964,675


In [24]:
batch = next(iter(train_loader))

print(type(batch))

<class 'dict'>


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x792b29993060>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x792b29993060>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

In [41]:
#  EMA + AMP TRAINING LOOP (WITH VISUAL & WEIGHT CHECKPOINTS)
import os
import copy
import torch
from tqdm import tqdm
from torchvision.utils import save_image, make_grid

# ==========================================
# 1. DIRECTORY & EMA SETUP
# ==========================================
print("something started in this cell")
progress_dir = "./training_progress"
os.makedirs(progress_dir, exist_ok=True)

class EMA:
    def __init__(self, beta=0.995):
        self.beta = beta
    
    def step_ema(self, ema_model, current_model):
        for ema_param, param in zip(ema_model.parameters(), current_model.parameters()):
            if param.requires_grad:
                ema_param.data.mul_(self.beta).add_(param.data, alpha=1.0 - self.beta)

# Create the shadow model BEFORE compiling the main model
ema_model = copy.deepcopy(model).eval().to(device)
ema_model.requires_grad_(False)
ema = EMA(beta=0.995)

# ==========================================
# 2. PYTORCH COMPILE & AMP SETUP
# ==========================================

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
loss_fn = torch.nn.MSELoss()
scaler = torch.amp.GradScaler('cuda')

epochs = 300
print(f"Initiating Training... Progress images and model checkpoints will be saved to '{progress_dir}' every 5 epochs.")

# ==========================================
# 3. TRAINING LOOP
# ==========================================

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    perm = torch.randperm(len(all_images_gpu), device=device)
    for step in tqdm(range(num_batches), desc=f"Epoch {epoch+1}/{epochs}"):
        images = all_images_gpu[perm[step * batch_size : (step + 1) * batch_size]]
        
        # Batch-level random flip on GPU (replaces per-sample CPU flip)
        flip_mask = torch.rand(images.shape[0], 1, 1, 1, device=device) > 0.5
        images = torch.where(flip_mask, images.flip(-1), images)
        current_batch = images.shape[0]
        
        t = torch.randint(0, timesteps, (current_batch,), device=device).long()
        true_noise = torch.randn_like(images)
        noisy_images = add_noise(images, true_noise, t, alpha_cumprod)
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast('cuda', dtype=torch.float16):
            pred_noise = model(noisy_images, t)
            loss = loss_fn(pred_noise, true_noise)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # Update the shadow EMA model every single step
        ema.step_ema(ema_model, model)
            
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {epoch_loss / num_batches:.6f}")

    # ==========================================
    # 4. CHECKPOINTING (Every 50 Epochs)
    # ==========================================
    if (epoch + 1) % 5 == 0:
        print(f"--> Generating progress grid and saving weights for Epoch {epoch+1}...")
        
        # --- A. Save the Model Weights ---
        checkpoint_path = os.path.join(progress_dir, f"ema_model_epoch_{epoch+1:03d}.pth")
        torch.save(ema_model.state_dict(), checkpoint_path)
        
        # --- B. Generate the Visual Grid ---
        ema_model.eval() 
        with torch.no_grad():
            x_gen = torch.randn((10, 3, 32, 32)).to(device) 
            
            for i in tqdm(reversed(range(timesteps)), total=timesteps, desc="Sampling", leave=False):
                t_gen = torch.full((10,), i, device=device, dtype=torch.long)
                pred_noise_gen = ema_model(x_gen, t_gen)
                
                a = alphas[t_gen][:, None, None, None]
                a_hat = alpha_cumprod[t_gen][:, None, None, None]
                b = betas[t_gen][:, None, None, None]
                
                z = torch.randn_like(x_gen) if i > 0 else torch.zeros_like(x_gen)
                x_gen = 1 / torch.sqrt(a) * (x_gen - ((1 - a) / (torch.sqrt(1 - a_hat))) * pred_noise_gen) + torch.sqrt(b) * z
            
            x_gen = (x_gen.clamp(-1, 1) + 1) / 2
            
            grid = make_grid(x_gen, nrow=5, padding=2, normalize=True)
            img_save_path = os.path.join(progress_dir, f"grid_epoch_{epoch+1:03d}.png")
            save_image(grid, img_save_path)

# Save final weights to the main directory
torch.save(ema_model.state_dict(), "ddpm_imageNet_EMA_weights.pth")

torch.save(model.state_dict(), "ddpm_imageNet_RAW_weights.pth")
    
print("\nTraining Complete! Final weights secured.")

something started in this cell
Initiating Training... Progress images and model checkpoints will be saved to './training_progress' every 5 epochs.


Epoch 1/300: 100%|██████████| 48/48 [00:46<00:00,  1.04it/s]


Epoch 1 | Loss: 0.635453


Epoch 2/300: 100%|██████████| 48/48 [00:43<00:00,  1.11it/s]


Epoch 2 | Loss: 0.268621


...

Epoch 299/300: 100%|██████████| 48/48 [00:44<00:00,  1.09it/s]


Epoch 299 | Loss: 0.107822


Epoch 300/300: 100%|██████████| 48/48 [00:44<00:00,  1.09it/s]


Epoch 300 | Loss: 0.107146
--> Generating progress grid and saving weights for Epoch 300...



Training Complete! Final weights secured.


In [42]:
import os
from torchvision.utils import save_image

# Settings
total_images_needed = 1000
gen_batch_size = 200 

@torch.no_grad()
def sample_images(diffusion_model, n_samples):
    diffusion_model.eval()
    x = torch.randn((n_samples, 3, 32, 32)).to(device)
    
    for i in tqdm(reversed(range(timesteps)), total=timesteps, desc=f"Sampling Batch", leave=False):
        t = torch.full((n_samples,), i, device=device, dtype=torch.long)
        predicted_noise = diffusion_model(x, t)
        
        alpha = alphas[t][:, None, None, None]
        alpha_hat = alpha_cumprod[t][:, None, None, None]
        beta = betas[t][:, None, None, None]
        
        noise = torch.randn_like(x) if i > 0 else torch.zeros_like(x)
        x = 1 / torch.sqrt(alpha) * (x - ((1 - alpha) / (torch.sqrt(1 - alpha_hat))) * predicted_noise) + torch.sqrt(beta) * noise
        
    return (x.clamp(-1, 1) + 1) / 2

gen_dir = "./evaluation_data/generated"
os.makedirs(gen_dir, exist_ok=True)

batches = total_images_needed // gen_batch_size
remainder = total_images_needed % gen_batch_size

print(f"\nGenerating {total_images_needed} images using EMA weights...")
img_counter = 0

# Use the ema_model for generation!
for b in range(batches):
    gen_imgs = sample_images(ema_model, n_samples=gen_batch_size) 
    for i in range(gen_batch_size):
        save_image(gen_imgs[i], os.path.join(gen_dir, f"img_{img_counter:05d}.png"))
        img_counter += 1

if remainder > 0:
    gen_imgs = sample_images(ema_model, n_samples=remainder)
    for i in range(remainder):
        save_image(gen_imgs[i], os.path.join(gen_dir, f"img_{img_counter:05d}.png"))
        img_counter += 1

print(f"Generation Complete! Dataset saved to '{gen_dir}'")


Generating 1000 images using EMA weights...


Generation Complete! Dataset saved to './evaluation_data/generated'


In [44]:
from IPython.display import FileLink
import os

# Zip the directory first if you haven't already
os.system('zip -r evaluation_data.zip /kaggle/working/evaluation_data')

# Generate the download link
FileLink(r'evaluation_data.zip')

  adding: kaggle/working/evaluation_data/ (stored 0%)
  adding: kaggle/working/evaluation_data/generated/ (stored 0%)
  adding: kaggle/working/evaluation_data/generated/img_00597.png (stored 0%)
  adding: kaggle/working/evaluation_data/generated/img_00360.png (stored 0%)
  adding: kaggle/working/evaluation_data/generated/img_00895.png (stored 0%)
  adding: kaggle/working/evaluation_data/generated/img_00324.png (stored 0%)
  adding: kaggle/working/evaluation_data/generated/img_00724.png (stored 0%)
  adding: kaggle/working/evaluation_data/generated/img_00407.png (stored 0%)
  adding: kaggle/working/evaluation_data/generated/img_00345.png (stored 0%)
  adding: kaggle/working/evaluation_data/generated/img_00207.png (stored 0%)
  adding: kaggle/working/evaluation_data/generated/img_00562.png (stored 0%)
  adding: kaggle/working/evaluation_data/generated/img_00824.png (stored 0%)
  adding: kaggle/working/evaluation_data/generated/img_00821.png (stored 0%)
  adding: kaggle/working/evaluation

/kaggle/working/evaluation_data.zip

In [45]:
pip install torch-fidelity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 3.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [47]:
# EVALUATION
import os
import torch
import torchvision.transforms as transforms
import torch_fidelity
from torch.utils.data import Dataset

class ImageOnlyDataset(Dataset):
    def __init__(self, images_tensor):
        # Un-normalize from ImageNet stats back to [0, 1]
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        self.images = (images_tensor.cpu() * std + mean).clamp(0, 1)
        # Convert to uint8 [0, 255] for torch_fidelity
        self.images = (self.images * 255).to(torch.uint8)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx]

evaluation_dataset = ImageOnlyDataset(all_images_gpu)

def evaluate_model(generated_images_dir: str, real_data):
    print("Executing memory-safe torch_fidelity diagnostic report...")
    
    metrics = torch_fidelity.calculate_metrics(
        input1=generated_images_dir,
        input2=real_data, 
        cuda=True,
        isc=True,
        fid=True,
        prc=False,
        save_cpu_ram=True,  
        verbose=False 
    )
    
    fid_score = metrics.get('frechet_inception_distance', 'N/A')
    is_mean = metrics.get('inception_score_mean', 'N/A')
    is_std = metrics.get('inception_score_std', 'N/A')

    print("=============================================")
    print("             DIAGNOSTIC REPORT")
    print("=============================================")
    if isinstance(fid_score, float): print(f"FID:       {fid_score:.4f}  (Lower is better)")
    if isinstance(is_mean, float): print(f"IS:        {is_mean:.4f} ± {is_std:.4f}  (Higher is better)")
    print("=============================================")

gen_dir = "./evaluation_data/generated"

evaluate_model(gen_dir, evaluation_dataset)

Executing memory-safe torch_fidelity diagnostic report...


Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth


             DIAGNOSTIC REPORT
FID:       86.5856  (Lower is better)
IS:        4.8901 ± 0.3931  (Higher is better)
